In [46]:
import boto3
import pandas as pd   
import os 
import numpy as np 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
import sagemaker 
from sagemaker.inputs import TrainingInput 
from sagemaker.tuner import HyperparameterTuner,IntegerParameter, ContinuousParameter
from sagemaker.estimator import Estimator
from sagemaker import image_uris 
from sagemaker.serializers import CSVSerializer


In [2]:
##S3 bucket and folder call

bucket_name = "bank-churn-visualpath-proj"
s3_prefix = 'data/'
files_to_upload = ['bank_doc_1.csv','bank_doc_2.csv']
s3 = boto3.client('s3')

for file_name in files_to_upload:
    s3_key = s3_prefix+file_name
    # print(f"s3://{bucket_name}/{s3_key}")
    s3.upload_file(file_name,bucket_name,s3_key)
    print(f"uploaded file {file_name} to s3://{bucket_name}/{s3_key}")

uploaded file bank_doc_1.csv to s3://bank-churn-visualpath-proj/data/bank_doc_1.csv
uploaded file bank_doc_2.csv to s3://bank-churn-visualpath-proj/data/bank_doc_2.csv


In [47]:
 df = pd.read_csv('s3://bank-churn-visualpath-proj/data/bank_doc_1.csv',sep=',')
 df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df['country'].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [48]:
 df_2 = pd.read_csv('s3://bank-churn-visualpath-proj/data/bank_doc_2.csv',sep=',')
 df_2.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15768359,534,France,Male,36,4,120037.96,1,1,0,36275.94,0
1,15805769,656,Spain,Male,33,4,0.00,2,1,0,116706.00,0
2,15719508,575,Germany,Male,49,7,121205.15,4,1,1,168080.53,1
3,15609011,480,Spain,Male,47,8,75408.33,1,1,0,25887.89,1
4,15703106,575,France,Male,40,5,0.00,2,1,1,122488.59,0


In [49]:
bank = pd.concat([df,df_2],ignore_index=True) 
bank.count()

customer_id         10000
credit_score        10000
country             10000
gender              10000
age                 10000
tenure              10000
balance             10000
products_number     10000
credit_card         10000
active_member       10000
estimated_salary    10000
churn               10000
dtype: int64

In [11]:
bank.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [13]:
bank.shape

(10000, 12)

In [50]:
X = bank.drop(['churn','customer_id'],axis=1)


In [8]:
X.shape

(10000, 10)

In [51]:
y = bank['churn']

In [52]:
X_train,X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=23,stratify=y)

In [11]:
X_train.shape,X_test.shape

((8000, 10), (2000, 10))

In [12]:
X_train.dtypes

credit_score          int64
country              object
gender               object
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
dtype: object

In [53]:
##Categorical columns and Numerical column split
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
num_cols = X_train.select_dtypes(exclude='object').columns.tolist() ##include=['int64','float64']


In [15]:
cat_cols

['country', 'gender']

In [16]:
num_cols

['credit_score',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary']

In [17]:
X_train[num_cols]

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
6768,599,42,6,0.00,2,1,0,113868.40
2346,518,46,4,113625.93,1,0,0,92727.42
4529,664,44,8,142989.69,1,1,1,115452.51
8166,542,37,8,0.00,1,1,1,807.06
1096,756,39,3,100717.85,3,1,1,73406.04
...,...,...,...,...,...,...,...,...
4242,526,50,5,124233.24,1,0,1,159456.87
9694,581,25,5,77886.53,2,1,0,150319.49
5089,688,45,9,103399.87,1,0,0,129870.93
1906,786,29,4,0.00,2,1,0,103372.79


In [54]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

In [15]:
X_train[num_cols]

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
6768,-0.532510,0.297673,0.333815,-1.215280,0.801843,0.646286,-1.023271,0.231695
2346,-1.370517,0.682809,-0.356867,0.601536,-0.917927,-1.547303,-1.023271,-0.134845
4529,0.139964,0.490241,1.024497,1.071046,-0.917927,0.646286,0.977259,0.259160
8166,-1.122219,-0.183746,1.024497,-1.215280,-0.917927,0.646286,0.977259,-1.728551
1096,1.091774,0.008822,-0.702208,0.395143,2.521613,0.646286,0.977259,-0.469838
...,...,...,...,...,...,...,...,...
4242,-1.287751,1.067944,-0.011526,0.771141,-0.917927,-1.547303,0.977259,1.022103
9694,-0.718734,-1.339151,-0.011526,0.030083,0.801843,0.646286,-1.023271,0.863680
5089,0.388262,0.586525,1.369838,0.438027,-0.917927,-1.547303,-1.023271,0.509145
1906,1.402146,-0.954016,-0.356867,-1.215280,0.801843,0.646286,-1.023271,0.049723


In [20]:
X_test[num_cols]

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
867,636,48,1,170833.46,1,1,0,110510.28
5295,706,29,6,185544.36,1,1,0,171037.63
3815,650,33,0,98064.97,1,1,0,52411.99
3866,547,29,6,104450.86,1,1,1,37160.28
5228,751,29,10,147737.63,1,0,1,94951.27
...,...,...,...,...,...,...,...,...
641,706,29,5,112564.62,1,1,0,42334.38
1960,655,36,1,135515.76,1,1,0,86013.96
1317,789,37,3,0.00,1,1,0,121883.87
744,650,60,8,0.00,2,1,1,102925.76


In [55]:
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [14]:
X_test[num_cols]

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
867,-0.149717,0.875376,-1.392890,1.516253,-0.917927,0.646286,-1.023271,0.173472
5295,0.574486,-0.954016,0.333815,1.751472,-0.917927,0.646286,-1.023271,1.222889
3815,-0.004877,-0.568881,-1.738231,0.352725,-0.917927,0.646286,-1.023271,-0.833830
3866,-1.070490,-0.954016,0.333815,0.454832,-0.917927,0.646286,0.977259,-1.098263
5228,1.040045,-0.954016,1.715179,1.146963,-0.917927,-1.547303,0.977259,-0.096289
...,...,...,...,...,...,...,...,...
641,0.574486,-0.954016,-0.011526,0.584566,-0.917927,0.646286,-1.023271,-1.008555
1960,0.046852,-0.280029,-1.392890,0.951542,-0.917927,0.646286,-1.023271,-0.251243
1317,1.433184,-0.183746,-0.702208,-1.215280,-0.917927,0.646286,-1.023271,0.370666
744,-0.004877,2.030782,1.024497,-1.215280,0.801843,0.646286,0.977259,0.041972


In [23]:
X_train[cat_cols]

,country,gender
6768,Spain,Male
2346,Germany,Male
4529,France,Female
8166,Spain,Male
1096,Germany,Female
...,...,...
4242,Germany,Male
9694,France,Male
5089,Germany,Male
1906,France,Female


In [95]:
X_train_enc = pd.get_dummies(X_train[cat_cols],drop_first=True).astype(int)

In [96]:
X_train_enc

,country_Germany,country_Spain,gender_Male
6768,0,1,1
2346,1,0,1
4529,0,0,0
8166,0,1,1
1096,1,0,0
...,...,...,...
4242,1,0,1
9694,0,0,1
5089,1,0,1
1906,0,0,0


In [94]:
X_test_enc = pd.get_dummies(X_test[cat_cols],drop_first=True).astype(int)

,country_Germany,country_Spain,gender_Male
867,0,0,0
5295,1,0,0
3815,0,0,1
3866,0,0,0
5228,0,0,1
...,...,...,...
641,0,0,0
1960,0,1,0
1317,0,0,1
744,0,0,1


In [33]:
X_train_enc

country_Germany    1977
country_Spain      1983
gender_Male        4352
dtype: int64

In [97]:
 X_train_transformed = pd.concat([X_train[num_cols],X_train_enc],axis=1)

In [98]:
 X_test_transformed = pd.concat([X_test[num_cols],X_test_enc],axis=1)

In [99]:
X_train_transformed

,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,country_Germany,country_Spain,gender_Male
6768,-0.532510,0.297673,0.333815,-1.215280,0.801843,0.646286,-1.023271,0.231695,0,1,1
2346,-1.370517,0.682809,-0.356867,0.601536,-0.917927,-1.547303,-1.023271,-0.134845,1,0,1
4529,0.139964,0.490241,1.024497,1.071046,-0.917927,0.646286,0.977259,0.259160,0,0,0
8166,-1.122219,-0.183746,1.024497,-1.215280,-0.917927,0.646286,0.977259,-1.728551,0,1,1
1096,1.091774,0.008822,-0.702208,0.395143,2.521613,0.646286,0.977259,-0.469838,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...
4242,-1.287751,1.067944,-0.011526,0.771141,-0.917927,-1.547303,0.977259,1.022103,1,0,1
9694,-0.718734,-1.339151,-0.011526,0.030083,0.801843,0.646286,-1.023271,0.863680,0,0,1
5089,0.388262,0.586525,1.369838,0.438027,-0.917927,-1.547303,-1.023271,0.509145,1,0,1
1906,1.402146,-0.954016,-0.356867,-1.215280,0.801843,0.646286,-1.023271,0.049723,0,0,0


In [100]:
##Create folder and store as csv

os.makedirs("bank_data",exist_ok=True) 
train_df = pd.concat([y_train.reset_index(drop=True),X_train_transformed.reset_index(drop=True)],axis=1)
test_df = pd.concat([y_test.reset_index(drop=True),X_test_transformed.reset_index(drop=True)],axis=1)

In [127]:
train_df.to_csv('bank_data/train.csv',index=False,header=False) 
test_df.to_csv('bank_data/test.csv',index=False,header=False)

In [128]:
prefix = 'xgboost_demo'
train_key = f'{prefix}/train/train.csv'
test_key = f'{prefix}/test/test.csv'


In [129]:
s3 = boto3.client('s3')
s3.upload_file('bank_data/train.csv','bank-churn-visualpath-proj',train_key)
s3.upload_file('bank_data/test.csv','bank-churn-visualpath-proj',test_key)

In [130]:
##Create session for sagemaker to perform
boto_session = boto3.Session(region_name='us-east-1') 
session = sagemaker.Session(boto_session=boto3.Session(region_name="us-east-1"))

In [131]:
bucket_name="bank-churn-visualpath-proj"

In [132]:
role = sagemaker.get_execution_role()


In [133]:
region = session.boto_region_name
print(region)

us-east-1


In [134]:
container = image_uris.retrieve(
    framework='xgboost',
    region=region, 
    version="1.7-1" 
)

In [135]:
#XGBoost estimator  - defining
xgb = Estimator(
    image_uri=container,
    role = role, 
    instance_count=1,
    instance_type = 'ml.m5.large', 
    output_path=f"s3://{bucket_name}/{prefix}/output",
    sagemaker_session=session)

In [136]:
xgb.set_hyperparameters(objective = "binary:logistic", num_round=100)

In [137]:
hyperparam_range = {
    "max_depth": IntegerParameter(3,10),
    "eta":ContinuousParameter(0.01,0.3),
    "min_child_weight":IntegerParameter(1,10), 
    "subsample":ContinuousParameter(.5,1.0), 
    "colsample_bytree": ContinuousParameter(0.5,1.0) 
}

In [141]:
tuner = HyperparameterTuner(
    estimator=xgb, 
    objective_metric_name="validation:auc", ##validation:logloss
    hyperparameter_ranges=hyperparam_range,
    metric_definitions= [
        { "Name": "validation:auc", 
          "Regex": "validation-auc:([0-9\\.]+)"
        }],
    max_jobs=10,
    max_parallel_jobs=2, 
    objective_type="Maximize" 
)

In [142]:
s3_train_path = "s3://bank-churn-visualpath-proj/xgboost_demo/train/train.csv"
s3_test_path = "s3://bank-churn-visualpath-proj/xgboost_demo/test/test.csv"

In [143]:
tuner.fit(
    {
        "train": TrainingInput(s3_train_path, content_type="text/csv"),
        "validation": TrainingInput(s3_test_path, content_type="text/csv")
    }
)

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


...........................................................................................!


In [144]:
best_estimator = tuner.best_estimator()



2025-11-30 06:43:38 Starting - Found matching resource for reuse
2025-11-30 06:43:38 Downloading - Downloading the training image
2025-11-30 06:43:38 Training - Training image download completed. Training in progress.
2025-11-30 06:43:38 Uploading - Uploading generated training model
2025-11-30 06:43:38 Completed - Resource reused by training job: sagemaker-xgboost-251130-0638-008-62dfa371


In [146]:
best_estimator.hyperparameters()

{'_tuning_objective_metric': 'validation:auc',
 'colsample_bytree': '0.6147763400471009',
 'eta': '0.014137692769491277',
 'max_depth': '9',
 'min_child_weight': '5',
 'num_round': '100',
 'objective': 'binary:logistic',
 'subsample': '0.8566487814277957'}

In [154]:
##Creating endpoint
predictor = best_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="bank-churn-xgb-endpoint"
)


-------!

In [155]:
payload = "619,42,2,0,1,1,1,101348.88,1,0,0,1,0"   # model-ready row


In [ ]:
## Send Prediction Request to Endpoint
result = predictor.predict(payload)
print(result)

In [ ]:
### Call Endpoint from AWS Lambda, API Gateway, or VS Code
##Below code is for VS Code you can use it.
from sagemaker.predictor import Predictor

predictor = Predictor(endpoint_name="bank-churn-xgb-endpoint")

result = predictor.predict(payload)


In [157]:
predictor.delete_endpoint() ##Once done delete the endpoint otherwise you will be charged
